# L10 – CNN och data augmentation

**Tid:** ca 2,5 h · **Del C: Nätverkstyper**

## Mål
1. Förstå **varför** en MLP passar dåligt för bilder och vad en CNN gör annorlunda.
2. Förstå **faltning** (convolution), **kernel**, **padding**, **stride** och **pooling** intuitivt, och kunna räkna ut shapes.
3. Bygga en CNN som en egen `nn.Module`-klass och träna den på CIFAR-10.
4. Använda **data augmentation**, och bara på träningsdata.

**Var i arbetsflödet?** Steg 5 (modell) och steg 8 (augmentation är regularisering). Arbetsflödet är detsamma som för MLP:n. Bara modellen och datapipelinen byts ut.

> Kräver `dlkit` från L09. **Körtid:** cirka 6–8 minuter totalt på GTX 1080.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from dlkit.seed import set_seed
from dlkit.train import fit, evaluate
from dlkit.plots import plot_history

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR = "../data"
loss_fn = nn.CrossEntropyLoss()

## 1. CIFAR-10

CIFAR-10 innehåller 60 000 **färgbilder** på 32×32 pixlar i 10 klasser: flygplan, bil, fågel, katt och så vidare. Uppdelningen är 50 000 för träning och 10 000 för test. Bilderna är mycket svårare än Fashion-MNIST, eftersom objekten syns från olika vinklar, i olika storlek och mot varierande bakgrund.

In [ ]:
raw = datasets.CIFAR10(DATA_DIR, train=True, download=True)
classes = raw.classes
print("data:", raw.data.shape, raw.data.dtype)   # (50000, 32, 32, 3): höjd, bredd, RGB

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for ax, i in zip(axes.flat, range(16)):
    ax.imshow(raw.data[i]); ax.set_title(classes[raw.targets[i]], fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

### Uppdelning och normalisering per kanal
Vi delar upp i 45 000 train och 5 000 val. Normaliseringen görs nu **per färgkanal**: tre medelvärden och tre standardavvikelser, beräknade på train (L04).

In [ ]:
rng = np.random.default_rng(42)
perm = rng.permutation(len(raw))
train_idx, val_idx = perm[:45_000], perm[45_000:]

train_pixels = raw.data[train_idx] / 255.0                  # (45000, 32, 32, 3)
MEAN = tuple(float(v) for v in train_pixels.mean(axis=(0, 1, 2)).round(4))   # ett värde per kanal
STD = tuple(float(v) for v in train_pixels.std(axis=(0, 1, 2)).round(4))
print("MEAN:", MEAN, " STD:", STD)

## 2. Varför inte en MLP?

En MLP plattar ut bilden till 32 × 32 × 3 = 3 072 tal. Det leder till två problem:
1. **Den vet inte att närliggande pixlar hör ihop.** Pixel 1 och pixel 2 är lika "långt ifrån" varandra som pixel 1 och pixel 3 000.
2. **Ett mönster som flyttas är ett helt nytt mönster.** En katt till vänster i bilden aktiverar helt andra vikter än en katt till höger. Modellen måste lära sig samma sak på varje plats.

Vi gör en baslinje med en MLP:

In [ ]:
eval_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

def make_loaders(train_tf, batch_size=128):
    # Två dataset-objekt över SAMMA filer: ett med träningstransform och ett med eval-transform.
    # Det är så man får augmentation på train men inte på val (se avsnitt 6).
    train_ds = Subset(datasets.CIFAR10(DATA_DIR, train=True, transform=train_tf), train_idx)
    val_ds = Subset(datasets.CIFAR10(DATA_DIR, train=True, transform=eval_tf), val_idx)
    return (DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4),
            DataLoader(val_ds, batch_size=256, shuffle=False, num_workers=4))

train_loader, val_loader = make_loaders(eval_tf)

set_seed(42)
mlp = nn.Sequential(nn.Flatten(), nn.Linear(3072, 512), nn.ReLU(),
                    nn.Linear(512, 256), nn.ReLU(), nn.Linear(256, 10)).to(device)
print(f"MLP: {sum(p.numel() for p in mlp.parameters()):,} parametrar")
h_mlp = fit(mlp, train_loader, val_loader, loss_fn,
            torch.optim.Adam(mlp.parameters(), lr=1e-3), device, epochs=8)

MLP:n når cirka **50 %**. Det är mycket bättre än slumpen (10 %) men långt från bra, trots 1,7 miljoner parametrar.

## 3. Faltning (convolution)

**Idé:** en liten **kernel** (filter), till exempel 3×3, glider över bilden. På varje position beräknas en viktad summa av pixlarna under kerneln. Resultatet blir en ny "bild", en **feature map**, som visar *var* i bilden kernelns mönster finns.

Det löser båda problemen:
1. **Lokalitet:** kerneln tittar bara på närliggande pixlar.
2. **Delade vikter:** *samma* kernel används på alla positioner. Ett mönster känns igen var det än finns, och antalet vikter blir litet.

I en CNN *lär sig* nätet kernlarna. Vi testar först en handgjord kernel som hittar vertikala kanter:

In [ ]:
img = torch.tensor(raw.data[7] / 255.0, dtype=torch.float32).mean(dim=2)   # gråskala, (32, 32)
vertical_edge = torch.tensor([[-1., 0., 1.],
                              [-2., 0., 2.],
                              [-1., 0., 1.]])                              # Sobel-filter
out = F.conv2d(img[None, None], vertical_edge[None, None], padding=1)       # shapes (N, C, H, W)

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(img, cmap="gray"); axes[0].set_title("indata")
axes[1].imshow(out[0, 0], cmap="gray"); axes[1].set_title("efter vertikalt kantfilter")
for ax in axes: ax.axis("off")
plt.show()

### Kernel, padding, stride och pooling

| Begrepp | Vad | Typiskt val |
|---|---|---|
| **Kernelstorlek** `k` | filtrets storlek | 3×3, litet och effektivt. Två 3×3 efter varandra "ser" lika mycket som en 5×5 men har färre vikter. |
| **Padding** `p` | nollor runt kanten | `p=1` för `k=3`, så att bilden behåller sin storlek ("same") |
| **Stride** `s` | steglängd när filtret glider | 1. Med 2 halveras storleken. |
| **Kanaler** | antal filter i ett lager, alltså antal feature maps ut | ökar djupare in i nätet: 32, 64, 128 |
| **Max pooling** | behåll största värdet i varje 2×2-ruta | halverar höjd och bredd. Billigt sätt att krympa bilden och tåla små förflyttningar. |

**Storlek ut:** $\text{ut} = \left\lfloor \frac{\text{in} - k + 2p}{s} \right\rfloor + 1$

In [ ]:
x = torch.randn(1, 3, 32, 32)          # en RGB-bild
for layer in [nn.Conv2d(3, 32, kernel_size=3, padding=1),
              nn.Conv2d(3, 32, kernel_size=3, padding=0),
              nn.Conv2d(3, 32, kernel_size=3, padding=1, stride=2),
              nn.MaxPool2d(2)]:
    print(f"{str(layer):70s} → {tuple(layer(x).shape)}")

**Parametrar:** `Conv2d(3, 32, 3)` har 32 filter med 3 × 3 × 3 vikter vardera plus 32 bias, alltså 896 parametrar, **oberoende av bildens storlek**. Ett `Linear`-lager från en 32×32×3-bild till 32 neuroner har 98 336.

**Mönstret i en CNN:** när vi går djupare blir bilden **mindre** (32, 16, 8, 4) och antalet **kanaler fler** (3, 32, 64, 128). De tidiga lagren hittar kanter och färger, mittlagren texturer och delar, och de sena lagren hela objektdelar.

## 4. Bygg en CNN som en `nn.Module`

Hittills har vi använt `nn.Sequential`. Mer flexibla modeller skrivs som en **klass**:
- `__init__` skapar lagren,
- `forward` beskriver hur datan flödar.

PyTorch hittar alla lager som sparas som attribut (`self.…`), så `model.parameters()` fungerar som vanligt.

In [ ]:
def conv_block(c_in, c_out):
    # Två 3x3-faltningar med BatchNorm (stabil träning i djupa nät, L07), sedan halvera storleken
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, kernel_size=3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
        nn.Conv2d(c_out, c_out, kernel_size=3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
        nn.MaxPool2d(2),
    )

class SmallCNN(nn.Module):
    def __init__(self, num_classes=10, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(3, 32),     # (3, 32, 32)  → (32, 16, 16)
            conv_block(32, 64),    # (32, 16, 16) → (64, 8, 8)
            conv_block(64, 128),   # (64, 8, 8)   → (128, 4, 4)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                       # 128 * 4 * 4 = 2048
            nn.Dropout(dropout),
            nn.Linear(128 * 4 * 4, num_classes),
        )

    def forward(self, x):
        x = self.features(x)       # "vad finns i bilden?"
        return self.classifier(x)  # "vilken klass är det?"

model = SmallCNN()
print(f"SmallCNN: {sum(p.numel() for p in model.parameters()):,} parametrar")
print("utdata:", model(torch.randn(2, 3, 32, 32)).shape)

CNN:en har cirka **5 gånger färre parametrar** än MLP:n. Mönstret är vanligt: `features` (faltningar) följt av `classifier` (ett litet MLP-huvud). Det är också den uppdelningen vi använder vid transfer learning i L11.

In [ ]:
set_seed(42)
cnn = SmallCNN().to(device)
h_cnn = fit(cnn, train_loader, val_loader, loss_fn,
            torch.optim.AdamW(cnn.parameters(), lr=1e-3, weight_decay=0.01), device, epochs=10)
plot_history(h_cnn, "SmallCNN utan augmentation")

CNN:en når cirka **80–83 %** på 10 epoker, jämfört med cirka 50 % för MLP:n. Men titta på kurvorna: **train accuracy drar ifrån val**. Gapet är cirka 8 procentenheter efter 10 epoker och växer om du tränar längre. Val loss sjunker allt långsammare och hackar. Det är början på överanpassning (L06), och här hjälper augmentation.

## 5. Data augmentation

**Idé:** skapa nya varianter av träningsbilderna med transformationer som **inte ändrar vilken klass bilden har**. En katt som speglas är fortfarande en katt. Modellen får se en ny variant varje epok och kan inte lära sig enskilda pixlar utantill. Det är "mer data" utan att samla in mer (L07, övning 3).

| Transformation | Vad | Lämplig när |
|---|---|---|
| `RandomHorizontalFlip()` | spegla vänster–höger | foton av objekt. **Inte** för text och siffror (b blir d). |
| `RandomCrop(32, padding=4)` | flytta bilden några pixlar | nästan alltid för små bilder |
| `ColorJitter(...)` | ändra ljusstyrka och kontrast | foton med varierande ljus |
| `RandomRotation(10)` | rotera lite | när objekt kan vara snedställda |

**Regeln:** augmentation ska bara göra sådant som *kan förekomma i verkligheten* för ditt problem.

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Visa samma bild med åtta olika slumpmässiga augmentations
show_tf = transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip()])
pil_img = datasets.CIFAR10(DATA_DIR, train=True)[7][0]
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax in axes:
    ax.imshow(show_tf(pil_img)); ax.axis("off")
plt.suptitle("En bild, åtta augmentations"); plt.show()

## 6. Augmentation bara på train

**Viktigt:** valideringsdatan och testdatan ska se ut som verklig data, utan slump. Annars mäter vi inte det vi tror, och resultaten blir brusiga.

Därför skapade `make_loaders` **två dataset-objekt** över samma filer med olika transforms. En vanlig bugg är att dela upp med `random_split` på *ett* dataset med augmentation. Då får även val augmentation.

In [ ]:
aug_train_loader, val_loader = make_loaders(train_tf)

set_seed(42)
cnn_aug = SmallCNN().to(device)
h_aug = fit(cnn_aug, aug_train_loader, val_loader, loss_fn,
            torch.optim.AdamW(cnn_aug.parameters(), lr=1e-3, weight_decay=0.01), device, epochs=15)
plot_history(h_aug, "SmallCNN med augmentation")

In [ ]:
for name, h in [("MLP", h_mlp), ("CNN", h_cnn), ("CNN + augmentation", h_aug)]:
    gap = h["train_acc"][-1] - h["val_acc"][-1]
    print(f"{name:20s} bästa val acc {max(h['val_acc']):.3f}   gap train-val (slut) {gap:+.3f}")

Med augmentation är **gapet mellan train och val mycket mindre**, och val accuracy blir högre. Träningen behöver fler epoker, eftersom uppgiften är svårare, men modellen generaliserar bättre.

## 7. Test – en gång

In [ ]:
test_ds = datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=eval_tf)
test_loader = DataLoader(test_ds, batch_size=256, num_workers=4)
print(f"TEST accuracy (CNN + augmentation): {evaluate(cnn_aug, test_loader, loss_fn, device)[1]:.3f}")

## 8. Övningar

### Övning 1 – räkna shapes
Räkna ut shape för utdata **för hand** och kontrollera sedan med koden.

In [ ]:
# TODO: fyll i dina svar som tupler (C, H, W)
answer_a = None   # nn.Conv2d(3, 16, kernel_size=5, padding=0) på (3, 32, 32)
answer_b = None   # nn.Conv2d(3, 16, kernel_size=3, padding=1, stride=2) på (3, 32, 32)
answer_c = None   # conv_block(3, 8) på (3, 28, 28)

In [ ]:
x = torch.randn(1, 3, 32, 32)
assert tuple(nn.Conv2d(3, 16, 5)(x).shape[1:]) == answer_a
assert tuple(nn.Conv2d(3, 16, 3, padding=1, stride=2)(x).shape[1:]) == answer_b
assert tuple(conv_block(3, 8)(torch.randn(1, 3, 28, 28)).shape[1:]) == answer_c
print("Rätt!")

### Övning 2 – räkna parametrar
Hur många parametrar har `nn.Conv2d(64, 128, kernel_size=3)`? Räkna för hand och kontrollera.

In [ ]:
my_count = None  # TODO

In [ ]:
assert my_count == sum(p.numel() for p in nn.Conv2d(64, 128, 3).parameters())
print("Rätt!", my_count)

### Övning 3 – en bredare CNN
Gör en egen variant av `SmallCNN` med fler kanaler (till exempel 64, 128 och 256) och träna den med augmentation i 15 epoker. Blev den bättre? Var det värt den längre träningstiden?

In [ ]:
# TODO

### Övning 4 – välj augmentation
Vilka augmentations är lämpliga, och vilka är **olämpliga**, för:
1. handskrivna siffror (MNIST),
2. röntgenbilder av lungor,
3. satellitbilder ovanifrån?

*Dina svar:*

## När använder jag detta?
- **CNN:** för bilder och annan data där närliggande värden hör ihop, till exempel spektrogram av ljud. För *riktiga* bildproblem med lite data börjar du hellre med en förtränad CNN (L11).
- **Byggsten:** `Conv 3×3`, `BatchNorm` och `ReLU` (×2), sedan `MaxPool`. Dubbla kanalerna varje gång bilden halveras.
- **Data augmentation:** nästan alltid för bilder. Välj transformationer som inte ändrar klassen. Bara på train.
- **`nn.Module`-klass** när modellen inte är en rak kedja, eller när du vill dela upp den i delar som `features` och `classifier`.

## Sammanfattning
- Faltning ger lokalitet och delade vikter, och därmed färre parametrar och bättre resultat på bilder.
- Shape-formeln: ut = ⌊(in − k + 2p)/s⌋ + 1.
- En liten CNN når cirka 80 % på CIFAR-10 jämfört med cirka 50 % för en MLP.
- Augmentation minskar överanpassning och höjer val accuracy.

**Nästa lektion:** transfer learning, att återanvända en CNN som tränats på miljontals bilder.